In [1]:
# 0. Data paths
path = '/data/storage/EXIST2023_training.json'
additional_path = '/data/storage/train_all_tasks.csv'
#test_path='/notebooks/Data/EXIST2023_test_clean.json'

In [2]:
# Standard Libraries
import json
import re
import time
import itertools

# Data Handling Libraries
import numpy as np
import pandas as pd

# Plotting Libraries
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning Libraries
from sklearn.model_selection import (
    StratifiedKFold, train_test_split,
    cross_val_score, GridSearchCV
)
from sklearn.preprocessing import (
    LabelEncoder, MultiLabelBinarizer, OneHotEncoder,
    StandardScaler, MinMaxScaler
)
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay,
    classification_report, label_ranking_average_precision_score
)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.multioutput import MultiOutputClassifier
from skmultilearn.model_selection import IterativeStratification
from imblearn.over_sampling import RandomOverSampler, SMOTE

# Natural Language Processing Libraries
import nltk
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from nlpaug.augmenter.char import RandomCharAug
from nlpaug.augmenter.word import SynonymAug, RandomWordAug

# Transformer Libraries
from transformers import (
    BertTokenizer, BertModel,
    DistilBertTokenizer, DistilBertModel,
    XLMRobertaTokenizer, XLMRobertaModel,
    GPT2Tokenizer
)

# PyTorch Libraries
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, random_split

# Optimization Libraries
import optuna

# GPU Monitoring
import GPUtil as GPU

# Nlpaug libraries
import nlpaug.augmenter.char as nac
import nlpaug.augmenter.word as naw
from nlpaug.augmenter.char import RandomCharAug
from nlpaug.augmenter.word import SynonymAug, RandomWordAug

# NLTK Downloads
nltk.download('wordnet')
nltk.download('punkt')
nltk.download('omw-1.4')


import torch
import torch.nn as nn
from transformers import BertModel, XLMRobertaModel, DistilBertModel, BertTokenizer
from torch.utils.data import DataLoader
import optuna
import time
import psutil
import GPUtil as GPU

/home/hmohammadi/.local/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[nltk_data] Downloading package wordnet to
[nltk_data]     /home/hmohammadi/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to /home/hmohammadi/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     /home/hmohammadi/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [3]:
#Functions
# Define a function to print system usage
def print_system_usage():
    print(f"CPU Usage: {psutil.cpu_percent()}%")
    print(f"RAM Usage: {psutil.virtual_memory().percent}%")
    GPUs = GPU.getGPUs()
    for i, gpu in enumerate(GPUs):
        print(f"GPU {i} Usage: {gpu.load*100:.1f}%")

# Function to get GPU memory usage
def get_gpu_memory():
    if torch.cuda.is_available():
        return torch.cuda.memory_allocated(device) / 1e9  # GBs
    return 0

def get_system_usage():
    ram_percent = psutil.virtual_memory().percent
    cpu_percent = psutil.cpu_percent()
    return ram_percent, cpu_percent

# Function to preprocess text
def preprocess_text(text):
    # Lowercase the text
    text = text.lower()
    # Remove URLs, special characters and punctuation
    text = re.sub(r'(https?://\S+|www\.\S+)', '', text)
    text = re.sub(r'[^a-z0-9\s]', '', text)

    # Tokenize the text
    words = word_tokenize(text)

    # Lemmatize words using WordNetLemmatizer
    lemmatizer = WordNetLemmatizer()
    lemmatized_words = [lemmatizer.lemmatize(word) for word in words]

    return " ".join(lemmatized_words)

def augment_text(text, num_augmentations=3):
    augmented_texts = []

    for _ in range(num_augmentations):
        # Apply the augmentations
        augmented_text = text
        augmented_text = aug_synonym.augment(augmented_text)
        augmented_text = aug_swap.augment(augmented_text)
        augmented_text = aug_insert.augment(augmented_text)

        # Append the augmented text to the list
        augmented_texts.append(augmented_text)

    return augmented_texts

# Compute the proportion of "YES" labels for each record
def preprocess_labels(labels):
    num_yes = sum([1 for label in labels if label == "YES"])
    proportion_yes = num_yes / len(labels)
    return proportion_yes

# Define a learning rate scheduler
def create_scheduler(learning_rate, warmup_steps, total_steps):
    def lr_lambda(epoch):
        step = epoch * (len(train_df_augmented['id_EXIST']) // 16)
        if step < warmup_steps:
            return learning_rate * (step / warmup_steps)
        return learning_rate * (0.5 * (1 + math.cos(math.pi * (step - warmup_steps) / (total_steps - warmup_steps))))
    return lr_lambda

def majority_voting(row):
    num_yes = sum(label == 'YES' for label in row)
    num_no = sum(label == 'NO' for label in row)
    return 'YES' if num_yes > num_no else 'NO'

In [4]:
# Clear GPU cache memory
torch.cuda.empty_cache()

# 3. GPU Setup
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Part 3
sample_size = 100
num_labels = 1
max_length = 256
MAX_LENGTH = 128
num_folds = 2

# batch_size_values = [128]
batch_size_values = [32, 64, 128]

# Set parameters
num_epochs = 5
batch_size = 64
learning_rate = 3e-05

# Load, preprocess, and combine data
with open(path) as f:
    data = json.load(f)

tweets = []
for tweet in data:
    if data[tweet]['labels_task1'] != []:
        lang = data[tweet]['lang']
        text = preprocess_text(data[tweet]['tweet'])
        label1 = data[tweet]['labels_task1']
        label2 = data[tweet]['labels_task2']
        label3 = data[tweet]['labels_task3']
        id = data[tweet]['id_EXIST']
        label4 = 'N/A'
        label5 = 'N/A'
        source = 'original'
        extra_info = '_'.join(map(str, [data[tweet]['number_annotators'], data[tweet]['annotators'][0],
                                        data[tweet]['gender_annotators'][0], data[tweet]['age_annotators'][0],
                                        data[tweet]['number_annotators'], data[tweet]['annotators'][1],
                                        data[tweet]['gender_annotators'][1], data[tweet]['age_annotators'][1],
                                        data[tweet]['number_annotators'], data[tweet]['annotators'][2],
                                        data[tweet]['gender_annotators'][2], data[tweet]['age_annotators'][2],
                                        data[tweet]['number_annotators'], data[tweet]['annotators'][3],
                                        data[tweet]['gender_annotators'][3], data[tweet]['age_annotators'][3],
                                        data[tweet]['number_annotators'], data[tweet]['annotators'][4],
                                        data[tweet]['gender_annotators'][4], data[tweet]['age_annotators'][4],
                                        data[tweet]['number_annotators'], data[tweet]['annotators'][5],
                                        data[tweet]['gender_annotators'][5], data[tweet]['age_annotators'][5]]))
        ...
        tweets.append((id, text, extra_info, label1, label2, label3, label4, label5, lang, source))

df = pd.DataFrame(tweets, columns=['id_EXIST', 'text', 'extra_info', 'label1', 'label2', 'label3', 'label4', 'label5', 'lang', 'source'])

df['proportion_yes'] = df.apply(lambda row: preprocess_labels(row['label1']), axis=1)

additional_df = pd.read_csv(additional_path)
additional_df['lang'] = 'en'
additional_df['source'] = 'additional'

additional_df = additional_df.rename(columns={'rewire_id': 'id', 'label_sexist': 'label1', 'label_category': 'label4', 'label_vector': 'label5'})
additional_df = additional_df.replace({'label1': {'sexist': 'YES', 'not sexist': 'NO'}})
additional_df['proportion_yes'] = additional_df['label1'].apply(lambda x: 1 if x == 'YES' else 0)

additional_df['label2'] = 'N/A'
additional_df['label3'] = 'N/A'
additional_df['extra_info'] = 'N/A'

additional_df = additional_df.rename(columns={'id': 'id_EXIST'})
additional_df['text'] = additional_df['text'].apply(preprocess_text)

#df = additional_df.iloc[0:sample_size, :]
#df = additional_df


# Part 4
aug_synonym = naw.SynonymAug(aug_src='wordnet')
aug_swap = naw.RandomWordAug(action="swap")
aug_insert = nac.RandomCharAug(action="insert")

train_df_augmented = df.copy()
train_data_augmented = []

for _, row in train_df_augmented.iterrows():
    text = row["text"]
    augmented_texts = augment_text(text)

    for augmented_text in augmented_texts:
        train_data_augmented.append({
            "id_EXIST": row["id_EXIST"],
            "text": augmented_text,
            "label1": row["label1"],
            "label2": row["label2"],
            "label3": row["label3"],
            "label4": row["label4"],
            "label5": row["label5"],
            "extra_info": row["extra_info"],
            "lang": row["lang"],
            "source": row["source"],
        })

Using device: cuda:0


In [5]:
df

,id_EXIST,text,extra_info,label1,label2,label3,label4,label5,lang,source,proportion_yes
0,100001,thechiflis ignora al otro e un capulloel probl...,6_Annotator_1_F_18-22_6_Annotator_2_F_23-45_6_...,"[YES, YES, NO, YES, YES, YES]","[REPORTED, JUDGEMENTAL, -, REPORTED, JUDGEMENT...","[[OBJECTIFICATION], [OBJECTIFICATION, SEXUAL-V...",N/A,N/A,es,original,0.833333
1,100002,ultimonomada si comicsgate se parece en algo a...,6_Annotator_7_F_18-22_6_Annotator_8_F_23-45_6_...,"[NO, NO, NO, NO, YES, NO]","[-, -, -, -, DIRECT, -]","[[-], [-], [-], [-], [OBJECTIFICATION], [-]]",N/A,N/A,es,original,0.166667
2,100003,steven2897 lee sobre gamergate y como eso ha c...,6_Annotator_7_F_18-22_6_Annotator_8_F_23-45_6_...,"[NO, NO, NO, NO, NO, NO]","[-, -, -, -, -, -]","[[-], [-], [-], [-], [-], [-]]",N/A,N/A,es,original,0.000000
3,100004,lunariita7 un retraso social bastante lamentab...,6_Annotator_13_F_18-22_6_Annotator_14_F_23-45_...,"[NO, NO, YES, NO, YES, YES]","[-, -, DIRECT, -, REPORTED, REPORTED]","[[-], [-], [IDEOLOGICAL-INEQUALITY], [-], [IDE...",N/A,N/A,es,original,0.500000
4,100005,novadragon21 icep4ck tvdannyz entonces como a ...,6_Annotator_19_F_18-22_6_Annotator_20_F_23-45_...,"[YES, NO, YES, NO, YES, YES]","[REPORTED, -, JUDGEMENTAL, -, JUDGEMENTAL, DIR...","[[STEREOTYPING-DOMINANCE, OBJECTIFICATION], [-...",N/A,N/A,es,original,0.666667
...,...,...,...,...,...,...,...,...,...,...,...
6915,203256,idk why yall bitch think having half your as h...,6_Annotator_478_F_18-22_6_Annotator_479_F_23-4...,"[YES, YES, YES, YES, YES, YES]","[JUDGEMENTAL, DIRECT, DIRECT, DIRECT, JUDGEMEN...","[[OBJECTIFICATION], [STEREOTYPING-DOMINANCE, S...",N/A,N/A,en,original,1.000000
6916,203257,this ha been a part of an experiment with word...,6_Annotator_668_F_18-22_6_Annotator_669_F_23-4...,"[YES, YES, YES, YES, YES, YES]","[JUDGEMENTAL, REPORTED, JUDGEMENTAL, DIRECT, J...","[[OBJECTIFICATION], [OBJECTIFICATION], [OBJECT...",N/A,N/A,en,original,1.000000
6917,203258,take me already not yet you got ta be readyim ...,6_Annotator_467_F_18-22_6_Annotator_468_F_23-4...,"[NO, YES, NO, YES, YES, YES]","[-, DIRECT, -, DIRECT, DIRECT, JUDGEMENTAL]","[[-], [OBJECTIFICATION], [-], [SEXUAL-VIOLENCE...",N/A,N/A,en,original,0.666667
6918,203259,clintneedcoffee why do you look like a whore lh,6_Annotator_674_F_18-22_6_Annotator_675_F_23-4...,"[YES, YES, YES, YES, YES, YES]","[DIRECT, DIRECT, DIRECT, DIRECT, JUDGEMENTAL, ...","[[OBJECTIFICATION, SEXUAL-VIOLENCE, MISOGYNY-N...",N/A,N/A,en,original,1.000000


In [7]:
train_df_augmented = pd.DataFrame(train_data_augmented)
train_df_augmented = train_df_augmented.sample(frac=1, random_state=42).reset_index(drop=True)

majority_labels = train_df_augmented['label1'].apply(majority_voting)
label_encoder = LabelEncoder()
binary_labels = label_encoder.fit_transform(majority_labels)
labels_as_binary = [1 if label.count('YES') > label.count('NO') else 0 for label in train_df_augmented['label1']]

# Part 5
X_train_fold, X_test_fold, y_train_fold, y_test_fold = train_test_split(train_df_augmented, labels_as_binary, test_size=0.2, stratify=labels_as_binary, random_state=42)

tokenizer = BertTokenizer.from_pretrained('bert-base-multilingual-uncased')
train_encodings = tokenizer(X_train_fold['text'].astype(str).tolist(), truncation=True, padding='max_length', max_length=256, return_tensors="pt")
test_encodings = tokenizer(X_test_fold['text'].astype(str).tolist(), truncation=True, padding='max_length', max_length=256, return_tensors="pt")

class CustomDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = CustomDataset(train_encodings, y_train_fold)
test_dataset = CustomDataset(test_encodings, y_test_fold)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

# Part 6
device = torch.device("cuda" if torch.cuda.is_available() else "CPU")

learning_rate = learning_rate
warmup_steps = 200
total_steps = num_epochs * (len(train_df_augmented['id_EXIST']) // 16)

In [8]:
# Train with the best hyperparameters

# Clear GPU cache memory
torch.cuda.empty_cache()

# Time to predict runtime
start_time = time.time()

class CustomBERTModel(nn.Module):
    def __init__(self, num_labels):
        super(CustomBERTModel, self).__init__()
        self.bert_tokenizer = BertTokenizer.from_pretrained('bert-base-multilingual-cased')
        self.bert_model = BertModel.from_pretrained('bert-base-multilingual-cased')

        self.xlm_roberta_tokenizer = XLMRobertaTokenizer.from_pretrained('xlm-roberta-base')
        self.xlm_roberta_model = XLMRobertaModel.from_pretrained('xlm-roberta-base')

        self.distilbert_tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-multilingual-cased')
        self.distilbert_model = DistilBertModel.from_pretrained('distilbert-base-multilingual-cased')

        # Concatenating the 3 models: bert, xlm_roberta, distilbert
        self.fc = nn.Linear(768 * 3, num_labels)

    #def forward(self, input_ids):
    def forward(self, input_ids, attention_mask=None, token_type_ids=None):
        
        bert_output = self.bert_model(input_ids).last_hidden_state
        xlm_roberta_output = self.xlm_roberta_model(input_ids).last_hidden_state
        distilbert_output = self.distilbert_model(input_ids).last_hidden_state

        concatenated = torch.cat((bert_output, xlm_roberta_output, distilbert_output), dim=2)
        out = self.fc(concatenated[:, 0, :])
        return out

GRADIENT_ACCUMULATION_STEPS = 2

def objective(trial):
    # Hyperparameters
    learning_rate = trial.suggest_float("learning_rate", 1e-5, 1e-3, log=True)
    batch_size = trial.suggest_int("batch_size", 64, 128, log=True)
    num_epochs = 3

    # Model, optimizer, criterion
    model = CustomBERTModel(num_labels=2).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
    criterion = nn.CrossEntropyLoss()

    # DataLoader
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0

        for batch_idx, batch in enumerate(train_loader):
            inputs = batch['input_ids'].to(device)
            labels = batch['labels'].to(device)

            # Clear gradients
            optimizer.zero_grad()

            # Forward pass
            outputs = model(inputs)
            loss = criterion(outputs, labels)

            # Backward pass
            loss.backward()

            # Gradient accumulation
            if (batch_idx + 1) % GRADIENT_ACCUMULATION_STEPS == 0:
                optimizer.step()
                optimizer.zero_grad()

            running_loss += loss.item()

        train_loss = running_loss / len(train_loader)

        # Clean up
        del inputs, labels, outputs
        torch.cuda.empty_cache()

    return train_loss

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=3)

print(study.best_params)

elapsed_time = time.time() - start_time
print(f"Estimated runtime: {elapsed_time:.2f} seconds")

print_system_usage()

[I 2024-05-03 13:04:17,733] A new study created in memory with name: no-name-c17fa7ce-980b-4fde-8af1-194bafc2d3f1
/tmp/ipykernel_11793/2248072060.py:22: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
We strongly recommend passing in an `attention_mask` since your input_ids may be padded. See https://huggingface.co/docs/transformers/troubleshooting#incorrect-output-when-padding-tokens-arent-masked.
[I 2024-05-03 13:24:20,479] Trial 0 finished with value: 0.672619833521647 and parameters: {'learning_rate': 8.696074563220994e-05, 'batch_size': 114}. Best is trial 0 with value: 0.672619833521647.
[I 2024-05-03 13:44:40,515] Trial 1 finished with value: 0.6726957458183479 and parameters: {'learning_rate': 5.602301769819298e-05, 'batch_size': 69}. Best is trial 1 with 

{'learning_rate': 0.0002475243746581922, 'batch_size': 70}
Estimated runtime: 3621.77 seconds
CPU Usage: 4.3%
RAM Usage: 2.2%
GPU 0 Usage: 100.0%


In [9]:
# 1. Train with the Best Hyperparameters

from sklearn.metrics import precision_score, recall_score, f1_score


#best_params ={'learning_rate': 0.0005446082609034526, 'batch_size': 124}
best_params = study.best_params
learning_rate = best_params["learning_rate"]
batch_size = best_params["batch_size"]
num_epochs = 3

model = CustomBERTModel(num_labels=2).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
criterion = nn.CrossEntropyLoss()
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

model.train()

start_time = time.time()
for epoch in range(num_epochs):
    epoch_start_time = time.time()
    running_loss = 0.0

    for batch_idx, batch in enumerate(train_loader):
        inputs = batch['input_ids'].to(device)
        labels = batch['labels'].to(device)

        optimizer.zero_grad()

        outputs = model(inputs)
        loss = criterion(outputs, labels)

        loss.backward()

        if (batch_idx + 1) % GRADIENT_ACCUMULATION_STEPS == 0 or batch_idx == len(train_loader) - 1:
            optimizer.step()
            optimizer.zero_grad()

        running_loss += loss.item() * inputs.size(0)

    train_loss = running_loss / len(train_dataset)
    print(f'Epoch [{epoch+1}/{num_epochs}], Train Loss: {train_loss:.4f}')

    
    # Estimation of finishing time
    epoch_duration = time.time() - epoch_start_time
    remaining_epochs = num_epochs - (epoch + 1)
    estimated_time_left = epoch_duration * remaining_epochs
    print(f"Estimated time left for training: {estimated_time_left:.2f} seconds")

    # Monitor RAM, CPU, and GPU usage
    ram_percent, cpu_percent = get_system_usage()
    gpu_memory = get_gpu_memory()
    print(f"RAM Usage: {ram_percent}%, CPU Usage: {cpu_percent}%, GPU Memory Used: {gpu_memory} GB")

total_training_time = time.time() - start_time
print(f"Total training time: {total_training_time:.2f} seconds")


# 2. Save the Model

torch.save(model.state_dict(), "best_model.pth")

# 3. Load the Model

loaded_model = CustomBERTModel(num_labels=2).to(device)
loaded_model.load_state_dict(torch.load("best_model.pth"))
loaded_model.eval()  # Set the model to evaluation mode

# 4. Evaluate the Model (assuming you have test_dataset for evaluation)

test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# Initialize counters
tp, fp, fn, correct = 0, 0, 0, 0
total = 0

with torch.no_grad():
    for batch in test_loader:
        inputs = batch['input_ids'].to(device)
        labels = batch['labels'].to(device)

        outputs = loaded_model(inputs)
        _, predicted = torch.max(outputs.data, 1)

        # Update TP, FP, FN
        tp += ((predicted == 1) & (labels == 1)).sum().item()
        fp += ((predicted == 1) & (labels == 0)).sum().item()
        fn += ((predicted == 0) & (labels == 1)).sum().item()

        # Update correct predictions
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

# Calculate Precision, Recall, F1, and Accuracy
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
accuracy = correct / total if total > 0 else 0

print(f'Accuracy: {accuracy:.2f}')
print(f'Precision: {precision:.2f}')
print(f'Recall: {recall:.2f}')
print(f'F1 Score: {f1:.2f}')

/tmp/ipykernel_11793/2248072060.py:22: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}


Epoch [1/3], Train Loss: 0.7081
Estimated time left for training: 795.64 seconds
RAM Usage: 2.3%, CPU Usage: 4.4%, GPU Memory Used: 7.104131072 GB
Epoch [2/3], Train Loss: 0.6759
Estimated time left for training: 398.56 seconds
RAM Usage: 2.3%, CPU Usage: 4.4%, GPU Memory Used: 7.104131072 GB
Epoch [3/3], Train Loss: 0.6814
Estimated time left for training: 0.00 seconds
RAM Usage: 2.3%, CPU Usage: 4.4%, GPU Memory Used: 7.104131072 GB
Total training time: 1194.71 seconds
Accuracy: 0.61
Precision: 0.00
Recall: 0.00
F1 Score: 0.00


In [ ]:
#SHAP

In [8]:
#Only BERT

In [9]:
# Train with the best hyperparameters

# Clear GPU cache memory
torch.cuda.empty_cache()

# Time to predict runtime
start_time = time.time()

class CustomBERTModel(nn.Module):
    def __init__(self, num_labels):
        super(CustomBERTModel, self).__init__()
        self.bert_tokenizer = BertTokenizer.from_pretrained('bert-base-multilingual-cased')
        self.bert_model = BertModel.from_pretrained('bert-base-multilingual-cased')

        #self.xlm_roberta_tokenizer = XLMRobertaTokenizer.from_pretrained('xlm-roberta-base')
        #self.xlm_roberta_model = XLMRobertaModel.from_pretrained('xlm-roberta-base')

        #self.distilbert_tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-multilingual-cased')
        #self.distilbert_model = DistilBertModel.from_pretrained('distilbert-base-multilingual-cased')

        # Concatenating the 3 models: bert, xlm_roberta, distilbert
        self.fc = nn.Linear(768 * 1, num_labels)

    #def forward(self, input_ids):
    def forward(self, input_ids, attention_mask=None, token_type_ids=None):
        
        bert_output = self.bert_model(input_ids).last_hidden_state
        #xlm_roberta_output = self.xlm_roberta_model(input_ids).last_hidden_state
        #distilbert_output = self.distilbert_model(input_ids).last_hidden_state

        #concatenated = torch.cat((bert_output, xlm_roberta_output, distilbert_output), dim=2)
        out = self.fc(bert_output[:, 0, :])
        return out

GRADIENT_ACCUMULATION_STEPS = 2

def objective(trial):
    # Hyperparameters
    learning_rate = trial.suggest_float("learning_rate", 1e-5, 1e-3, log=True)
    batch_size = trial.suggest_int("batch_size", 64, 128, log=True)
    num_epochs = 3

    # Model, optimizer, criterion
    model = CustomBERTModel(num_labels=2).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
    criterion = nn.CrossEntropyLoss()

    # DataLoader
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0

        for batch_idx, batch in enumerate(train_loader):
            inputs = batch['input_ids'].to(device)
            labels = batch['labels'].to(device)

            # Clear gradients
            optimizer.zero_grad()

            # Forward pass
            outputs = model(inputs)
            loss = criterion(outputs, labels)

            # Backward pass
            loss.backward()

            # Gradient accumulation
            if (batch_idx + 1) % GRADIENT_ACCUMULATION_STEPS == 0:
                optimizer.step()
                optimizer.zero_grad()

            running_loss += loss.item()

        train_loss = running_loss / len(train_loader)

        # Clean up
        del inputs, labels, outputs
        torch.cuda.empty_cache()

    return train_loss

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=3)

print(study.best_params)

elapsed_time = time.time() - start_time
print(f"Estimated runtime: {elapsed_time:.2f} seconds")

print_system_usage()

[I 2024-04-11 18:58:21,300] A new study created in memory with name: no-name-22ab24b4-e795-47b9-92ec-cd9888aa761f
/tmp/ipykernel_465641/2248072060.py:22: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
[I 2024-04-11 18:58:22,634] Trial 0 finished with value: 0.5468849539756775 and parameters: {'learning_rate': 2.3417707287537145e-05, 'batch_size': 114}. Best is trial 0 with value: 0.5468849539756775.
[I 2024-04-11 18:58:23,935] Trial 1 finished with value: 0.6693851947784424 and parameters: {'learning_rate': 0.0004645561539193115, 'batch_size': 98}. Best is trial 1 with value: 0.6693851947784424.
[I 2024-04-11 18:58:25,342] Trial 2 finished with value: 0.561237096786499 and parameters: {'learning_rate': 2.9087413947926173e-05, 'batch_size': 64}. Best is trial 1 wi

{'learning_rate': 0.0004645561539193115, 'batch_size': 98}
Estimated runtime: 4.04 seconds
CPU Usage: 2.3%
RAM Usage: 2.9%
GPU 0 Usage: 99.0%


In [10]:
# 1. Train with the Best Hyperparameters

from sklearn.metrics import precision_score, recall_score, f1_score


#best_params ={'learning_rate': 0.0005446082609034526, 'batch_size': 124}
best_params = study.best_params
learning_rate = best_params["learning_rate"]
batch_size = best_params["batch_size"]
num_epochs = 3

model = CustomBERTModel(num_labels=2).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
criterion = nn.CrossEntropyLoss()
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

model.train()

start_time = time.time()
for epoch in range(num_epochs):
    epoch_start_time = time.time()
    running_loss = 0.0

    for batch_idx, batch in enumerate(train_loader):
        inputs = batch['input_ids'].to(device)
        labels = batch['labels'].to(device)

        optimizer.zero_grad()

        outputs = model(inputs)
        loss = criterion(outputs, labels)

        loss.backward()

        if (batch_idx + 1) % GRADIENT_ACCUMULATION_STEPS == 0 or batch_idx == len(train_loader) - 1:
            optimizer.step()
            optimizer.zero_grad()

        running_loss += loss.item() * inputs.size(0)

    train_loss = running_loss / len(train_dataset)
    print(f'Epoch [{epoch+1}/{num_epochs}], Train Loss: {train_loss:.4f}')

    
    # Estimation of finishing time
    epoch_duration = time.time() - epoch_start_time
    remaining_epochs = num_epochs - (epoch + 1)
    estimated_time_left = epoch_duration * remaining_epochs
    print(f"Estimated time left for training: {estimated_time_left:.2f} seconds")

    # Monitor RAM, CPU, and GPU usage
    ram_percent, cpu_percent = get_system_usage()
    gpu_memory = get_gpu_memory()
    print(f"RAM Usage: {ram_percent}%, CPU Usage: {cpu_percent}%, GPU Memory Used: {gpu_memory} GB")

total_training_time = time.time() - start_time
print(f"Total training time: {total_training_time:.2f} seconds")


# 2. Save the Model

torch.save(model.state_dict(), "best_model_bert.pth")

# 3. Load the Model

loaded_model = CustomBERTModel(num_labels=2).to(device)
loaded_model.load_state_dict(torch.load("best_model_bert.pth"))
loaded_model.eval()  # Set the model to evaluation mode

# 4. Evaluate the Model (assuming you have test_dataset for evaluation)

test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# Initialize counters
tp, fp, fn, correct = 0, 0, 0, 0
total = 0

with torch.no_grad():
    for batch in test_loader:
        inputs = batch['input_ids'].to(device)
        labels = batch['labels'].to(device)

        outputs = loaded_model(inputs)
        _, predicted = torch.max(outputs.data, 1)

        # Update TP, FP, FN
        tp += ((predicted == 1) & (labels == 1)).sum().item()
        fp += ((predicted == 1) & (labels == 0)).sum().item()
        fn += ((predicted == 0) & (labels == 1)).sum().item()

        # Update correct predictions
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

# Calculate Precision, Recall, F1, and Accuracy
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
accuracy = correct / total if total > 0 else 0

print(f'Accuracy: {accuracy:.2f}')
print(f'Precision: {precision:.2f}')
print(f'Recall: {recall:.2f}')
print(f'F1 Score: {f1:.2f}')

/tmp/ipykernel_465641/2248072060.py:22: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}


Epoch [1/3], Train Loss: 0.5452
Estimated time left for training: 0.48 seconds
RAM Usage: 3.0%, CPU Usage: 8.8%, GPU Memory Used: 4.510380544 GB
Epoch [2/3], Train Loss: 0.3112
Estimated time left for training: 0.24 seconds
RAM Usage: 3.0%, CPU Usage: 4.2%, GPU Memory Used: 4.510380544 GB
Epoch [3/3], Train Loss: 0.8247
Estimated time left for training: 0.00 seconds
RAM Usage: 3.0%, CPU Usage: 4.0%, GPU Memory Used: 4.510380544 GB
Total training time: 0.72 seconds
Accuracy: 0.83
Precision: 0.00
Recall: 0.00
F1 Score: 0.00


In [11]:
# xlmOnly 

In [12]:
# Train with the best hyperparameters

# Clear GPU cache memory
torch.cuda.empty_cache()

# Time to predict runtime
start_time = time.time()

class CustomBERTModel(nn.Module):
    def __init__(self, num_labels):
        super(CustomBERTModel, self).__init__()
        #self.bert_tokenizer = BertTokenizer.from_pretrained('bert-base-multilingual-cased')
        #self.bert_model = BertModel.from_pretrained('bert-base-multilingual-cased')

        self.xlm_roberta_tokenizer = XLMRobertaTokenizer.from_pretrained('xlm-roberta-base')
        self.xlm_roberta_model = XLMRobertaModel.from_pretrained('xlm-roberta-base')

        #self.distilbert_tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-multilingual-cased')
        #self.distilbert_model = DistilBertModel.from_pretrained('distilbert-base-multilingual-cased')

        # Concatenating the 3 models: bert, xlm_roberta, distilbert
        self.fc = nn.Linear(768 * 1, num_labels)

    #def forward(self, input_ids):
    def forward(self, input_ids, attention_mask=None, token_type_ids=None):
        
        #bert_output = self.bert_model(input_ids).last_hidden_state
        xlm_roberta_output = self.xlm_roberta_model(input_ids).last_hidden_state
        #distilbert_output = self.distilbert_model(input_ids).last_hidden_state

        #concatenated = torch.cat((bert_output, xlm_roberta_output, distilbert_output), dim=2)
        out = self.fc(xlm_roberta_output[:, 0, :])
        return out

GRADIENT_ACCUMULATION_STEPS = 2

def objective(trial):
    # Hyperparameters
    learning_rate = trial.suggest_float("learning_rate", 1e-5, 1e-3, log=True)
    batch_size = trial.suggest_int("batch_size", 64, 128, log=True)
    num_epochs = 3

    # Model, optimizer, criterion
    model = CustomBERTModel(num_labels=2).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
    criterion = nn.CrossEntropyLoss()

    # DataLoader
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0

        for batch_idx, batch in enumerate(train_loader):
            inputs = batch['input_ids'].to(device)
            labels = batch['labels'].to(device)

            # Clear gradients
            optimizer.zero_grad()

            # Forward pass
            outputs = model(inputs)
            loss = criterion(outputs, labels)

            # Backward pass
            loss.backward()

            # Gradient accumulation
            if (batch_idx + 1) % GRADIENT_ACCUMULATION_STEPS == 0:
                optimizer.step()
                optimizer.zero_grad()

            running_loss += loss.item()

        train_loss = running_loss / len(train_loader)

        # Clean up
        del inputs, labels, outputs
        torch.cuda.empty_cache()

    return train_loss

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=3)

print(study.best_params)

elapsed_time = time.time() - start_time
print(f"Estimated runtime: {elapsed_time:.2f} seconds")

print_system_usage()

[I 2024-04-11 18:58:34,099] A new study created in memory with name: no-name-670b89a4-fb2b-4347-bc3e-9d1ed608a12c
/tmp/ipykernel_465641/2248072060.py:22: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
[I 2024-04-11 18:58:36,429] Trial 0 finished with value: 0.41271114349365234 and parameters: {'learning_rate': 0.0003626268228836553, 'batch_size': 64}. Best is trial 0 with value: 0.41271114349365234.
[I 2024-04-11 18:58:38,777] Trial 1 finished with value: 1.0235832929611206 and parameters: {'learning_rate': 8.397940696032987e-05, 'batch_size': 82}. Best is trial 1 with value: 1.0235832929611206.
[I 2024-04-11 18:58:41,047] Trial 2 finished with value: 0.44268786907196045 and parameters: {'learning_rate': 8.723266473097063e-05, 'batch_size': 65}. Best is trial 1 w

{'learning_rate': 8.397940696032987e-05, 'batch_size': 82}
Estimated runtime: 6.95 seconds
CPU Usage: 3.8%
RAM Usage: 3.0%
GPU 0 Usage: 98.0%


In [13]:
# 1. Train with the Best Hyperparameters

from sklearn.metrics import precision_score, recall_score, f1_score


#best_params ={'learning_rate': 0.0005446082609034526, 'batch_size': 124}
best_params = study.best_params
learning_rate = best_params["learning_rate"]
batch_size = best_params["batch_size"]
num_epochs = 3

model = CustomBERTModel(num_labels=2).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
criterion = nn.CrossEntropyLoss()
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

model.train()

start_time = time.time()
for epoch in range(num_epochs):
    epoch_start_time = time.time()
    running_loss = 0.0

    for batch_idx, batch in enumerate(train_loader):
        inputs = batch['input_ids'].to(device)
        labels = batch['labels'].to(device)

        optimizer.zero_grad()

        outputs = model(inputs)
        loss = criterion(outputs, labels)

        loss.backward()

        if (batch_idx + 1) % GRADIENT_ACCUMULATION_STEPS == 0 or batch_idx == len(train_loader) - 1:
            optimizer.step()
            optimizer.zero_grad()

        running_loss += loss.item() * inputs.size(0)

    train_loss = running_loss / len(train_dataset)
    print(f'Epoch [{epoch+1}/{num_epochs}], Train Loss: {train_loss:.4f}')

    
    # Estimation of finishing time
    epoch_duration = time.time() - epoch_start_time
    remaining_epochs = num_epochs - (epoch + 1)
    estimated_time_left = epoch_duration * remaining_epochs
    print(f"Estimated time left for training: {estimated_time_left:.2f} seconds")

    # Monitor RAM, CPU, and GPU usage
    ram_percent, cpu_percent = get_system_usage()
    gpu_memory = get_gpu_memory()
    print(f"RAM Usage: {ram_percent}%, CPU Usage: {cpu_percent}%, GPU Memory Used: {gpu_memory} GB")

total_training_time = time.time() - start_time
print(f"Total training time: {total_training_time:.2f} seconds")


# 2. Save the Model

torch.save(model.state_dict(), "best_model_xlm.pth")

# 3. Load the Model

loaded_model = CustomBERTModel(num_labels=2).to(device)
loaded_model.load_state_dict(torch.load("best_model_xlm.pth"))
loaded_model.eval()  # Set the model to evaluation mode

# 4. Evaluate the Model (assuming you have test_dataset for evaluation)

test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# Initialize counters
tp, fp, fn, correct = 0, 0, 0, 0
total = 0

with torch.no_grad():
    for batch in test_loader:
        inputs = batch['input_ids'].to(device)
        labels = batch['labels'].to(device)

        outputs = loaded_model(inputs)
        _, predicted = torch.max(outputs.data, 1)

        # Update TP, FP, FN
        tp += ((predicted == 1) & (labels == 1)).sum().item()
        fp += ((predicted == 1) & (labels == 0)).sum().item()
        fn += ((predicted == 0) & (labels == 1)).sum().item()

        # Update correct predictions
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

# Calculate Precision, Recall, F1, and Accuracy
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
accuracy = correct / total if total > 0 else 0

print(f'Accuracy: {accuracy:.2f}')
print(f'Precision: {precision:.2f}')
print(f'Recall: {recall:.2f}')
print(f'F1 Score: {f1:.2f}')

/tmp/ipykernel_465641/2248072060.py:22: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}


Epoch [1/3], Train Loss: 0.8215
Estimated time left for training: 0.49 seconds
RAM Usage: 2.9%, CPU Usage: 7.2%, GPU Memory Used: 4.0631936 GB
Epoch [2/3], Train Loss: 0.6273
Estimated time left for training: 0.24 seconds
RAM Usage: 2.9%, CPU Usage: 4.3%, GPU Memory Used: 4.0631936 GB
Epoch [3/3], Train Loss: 0.5513
Estimated time left for training: 0.00 seconds
RAM Usage: 2.9%, CPU Usage: 4.6%, GPU Memory Used: 4.0631936 GB
Total training time: 0.73 seconds
Accuracy: 0.83
Precision: 0.00
Recall: 0.00
F1 Score: 0.00


In [14]:
#only distilbert

In [15]:
# Train with the best hyperparameters

# Clear GPU cache memory
torch.cuda.empty_cache()

# Time to predict runtime
start_time = time.time()

class CustomBERTModel(nn.Module):
    def __init__(self, num_labels):
        super(CustomBERTModel, self).__init__()
        #self.bert_tokenizer = BertTokenizer.from_pretrained('bert-base-multilingual-cased')
        #self.bert_model = BertModel.from_pretrained('bert-base-multilingual-cased')

        #self.xlm_roberta_tokenizer = XLMRobertaTokenizer.from_pretrained('xlm-roberta-base')
        #self.xlm_roberta_model = XLMRobertaModel.from_pretrained('xlm-roberta-base')

        self.distilbert_tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-multilingual-cased')
        self.distilbert_model = DistilBertModel.from_pretrained('distilbert-base-multilingual-cased')

        # Concatenating the 3 models: bert, xlm_roberta, distilbert
        self.fc = nn.Linear(768 * 1, num_labels)

    #def forward(self, input_ids):
    def forward(self, input_ids, attention_mask=None, token_type_ids=None):
        
        #bert_output = self.bert_model(input_ids).last_hidden_state
        #xlm_roberta_output = self.xlm_roberta_model(input_ids).last_hidden_state
        distilbert_output = self.distilbert_model(input_ids).last_hidden_state

        #concatenated = torch.cat((bert_output, xlm_roberta_output, distilbert_output), dim=2)
        out = self.fc(distilbert_output[:, 0, :])
        return out

GRADIENT_ACCUMULATION_STEPS = 2

def objective(trial):
    # Hyperparameters
    learning_rate = trial.suggest_float("learning_rate", 1e-5, 1e-3, log=True)
    batch_size = trial.suggest_int("batch_size", 64, 128, log=True)
    num_epochs = 3

    # Model, optimizer, criterion
    model = CustomBERTModel(num_labels=2).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
    criterion = nn.CrossEntropyLoss()

    # DataLoader
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0

        for batch_idx, batch in enumerate(train_loader):
            inputs = batch['input_ids'].to(device)
            labels = batch['labels'].to(device)

            # Clear gradients
            optimizer.zero_grad()

            # Forward pass
            outputs = model(inputs)
            loss = criterion(outputs, labels)

            # Backward pass
            loss.backward()

            # Gradient accumulation
            if (batch_idx + 1) % GRADIENT_ACCUMULATION_STEPS == 0:
                optimizer.step()
                optimizer.zero_grad()

            running_loss += loss.item()

        train_loss = running_loss / len(train_loader)

        # Clean up
        del inputs, labels, outputs
        torch.cuda.empty_cache()

    return train_loss

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=3)

print(study.best_params)

elapsed_time = time.time() - start_time
print(f"Estimated runtime: {elapsed_time:.2f} seconds")

print_system_usage()

[I 2024-04-11 18:58:45,956] A new study created in memory with name: no-name-4673d8d6-9061-42a6-b0ae-2b2406b00961
/tmp/ipykernel_465641/2248072060.py:22: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
[I 2024-04-11 18:58:46,849] Trial 0 finished with value: 0.6613600850105286 and parameters: {'learning_rate': 2.17044174896306e-05, 'batch_size': 83}. Best is trial 0 with value: 0.6613600850105286.
[I 2024-04-11 18:58:47,703] Trial 1 finished with value: 0.6469465494155884 and parameters: {'learning_rate': 0.00013750333068961415, 'batch_size': 124}. Best is trial 0 with value: 0.6613600850105286.
[I 2024-04-11 18:58:48,561] Trial 2 finished with value: 0.7223629951477051 and parameters: {'learning_rate': 1.2783866920775571e-05, 'batch_size': 108}. Best is trial 2 w

{'learning_rate': 1.2783866920775571e-05, 'batch_size': 108}
Estimated runtime: 2.61 seconds
CPU Usage: 6.2%
RAM Usage: 2.9%
GPU 0 Usage: 100.0%


In [16]:
# 1. Train with the Best Hyperparameters

from sklearn.metrics import precision_score, recall_score, f1_score


#best_params ={'learning_rate': 0.0005446082609034526, 'batch_size': 124}
best_params = study.best_params
learning_rate = best_params["learning_rate"]
batch_size = best_params["batch_size"]
num_epochs = 3

model = CustomBERTModel(num_labels=2).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
criterion = nn.CrossEntropyLoss()
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

model.train()

start_time = time.time()
for epoch in range(num_epochs):
    epoch_start_time = time.time()
    running_loss = 0.0

    for batch_idx, batch in enumerate(train_loader):
        inputs = batch['input_ids'].to(device)
        labels = batch['labels'].to(device)

        optimizer.zero_grad()

        outputs = model(inputs)
        loss = criterion(outputs, labels)

        loss.backward()

        if (batch_idx + 1) % GRADIENT_ACCUMULATION_STEPS == 0 or batch_idx == len(train_loader) - 1:
            optimizer.step()
            optimizer.zero_grad()

        running_loss += loss.item() * inputs.size(0)

    train_loss = running_loss / len(train_dataset)
    print(f'Epoch [{epoch+1}/{num_epochs}], Train Loss: {train_loss:.4f}')

    
    # Estimation of finishing time
    epoch_duration = time.time() - epoch_start_time
    remaining_epochs = num_epochs - (epoch + 1)
    estimated_time_left = epoch_duration * remaining_epochs
    print(f"Estimated time left for training: {estimated_time_left:.2f} seconds")

    # Monitor RAM, CPU, and GPU usage
    ram_percent, cpu_percent = get_system_usage()
    gpu_memory = get_gpu_memory()
    print(f"RAM Usage: {ram_percent}%, CPU Usage: {cpu_percent}%, GPU Memory Used: {gpu_memory} GB")

total_training_time = time.time() - start_time
print(f"Total training time: {total_training_time:.2f} seconds")


# 2. Save the Model

torch.save(model.state_dict(), "best_model_distil.pth")

# 3. Load the Model

loaded_model = CustomBERTModel(num_labels=2).to(device)
loaded_model.load_state_dict(torch.load("best_model_distil.pth"))
loaded_model.eval()  # Set the model to evaluation mode

# 4. Evaluate the Model (assuming you have test_dataset for evaluation)

test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# Initialize counters
tp, fp, fn, correct = 0, 0, 0, 0
total = 0

with torch.no_grad():
    for batch in test_loader:
        inputs = batch['input_ids'].to(device)
        labels = batch['labels'].to(device)

        outputs = loaded_model(inputs)
        _, predicted = torch.max(outputs.data, 1)

        # Update TP, FP, FN
        tp += ((predicted == 1) & (labels == 1)).sum().item()
        fp += ((predicted == 1) & (labels == 0)).sum().item()
        fn += ((predicted == 0) & (labels == 1)).sum().item()

        # Update correct predictions
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

# Calculate Precision, Recall, F1, and Accuracy
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
accuracy = correct / total if total > 0 else 0

print(f'Accuracy: {accuracy:.2f}')
print(f'Precision: {precision:.2f}')
print(f'Recall: {recall:.2f}')
print(f'F1 Score: {f1:.2f}')

/tmp/ipykernel_465641/2248072060.py:22: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}


Epoch [1/3], Train Loss: 0.5372
Estimated time left for training: 0.25 seconds
RAM Usage: 3.0%, CPU Usage: 6.3%, GPU Memory Used: 2.748589568 GB
Epoch [2/3], Train Loss: 0.3393
Estimated time left for training: 0.12 seconds
RAM Usage: 3.0%, CPU Usage: 4.4%, GPU Memory Used: 2.748589568 GB
Epoch [3/3], Train Loss: 0.3006
Estimated time left for training: 0.00 seconds
RAM Usage: 3.0%, CPU Usage: 4.1%, GPU Memory Used: 2.748589568 GB
Total training time: 0.37 seconds
Accuracy: 0.83
Precision: 0.00
Recall: 0.00
F1 Score: 0.00


In [17]:
print('END of code')

END of code


In [18]:
#Explaianable part

In [8]:
model = CustomBERTModel(num_labels=2)
model.load_state_dict(torch.load("best_model.pth"))
model.eval()

CustomBERTModel(
  (bert_model): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(119547, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elemen

In [10]:
# It is safer to use strict=False or handle missing/unexpected keys explicitly
model.load_state_dict(torch.load("best_model.pth"), strict=False)

<All keys matched successfully>

In [11]:
forward_func = lambda i_ids, att_mask: model(i_ids, attention_mask=att_mask, token_type_ids=None)

In [12]:
# Assuming the model and tokenizer are already initialized
input_text = "Example text for explanation."
inputs = tokenizer(input_text, return_tensors="pt", add_special_tokens=True)
input_ids = inputs['input_ids']
attention_mask = inputs['attention_mask']

# Correct type should be ensured right after tokenization
input_ids = input_ids.type(torch.LongTensor)
attention_mask = attention_mask.type(torch.LongTensor)

label = 0  # Assuming binary classification and you're interested in class '0'
attributions = compute_attributions(model, input_ids, attention_mask, label)

# Display attributions
tokens = tokenizer.convert_ids_to_tokens(input_ids.squeeze().tolist())
attributions = attributions.squeeze().tolist()  # Assuming the output is correctly shaped
for token, attribution in zip(tokens, attributions):
    print(f"Token: {token} \t Attribution: {attribution:.4f}")

Input IDs type before unsqueeze: torch.int64
Input IDs type after unsqueeze: torch.int64


ValueError: too many values to unpack (expected 2)

In [14]:
# Ensure model's forward function returns logits
def forward(self, input_ids, attention_mask=None, token_type_ids=None):
    # Original forward code
    # Ensure that the output is [batch_size, num_labels]
    logits = self.fc(concatenated[:, 0, :])
    return logits  # Make sure to return logits

# Function to compute attributions using Integrated Gradients
def compute_attributions(model, input_ids, attention_mask, label):
    input_ids = input_ids.unsqueeze(0)  # Ensure batch dimension is present
    attention_mask = attention_mask.unsqueeze(0) if attention_mask is not None else None

    baseline_ids = torch.zeros_like(input_ids).long()
    baseline_mask = torch.zeros_like(attention_mask).long() if attention_mask is not None else None

    label = torch.tensor([label]).long()

    # Captum's IntegratedGradients
    integrated_gradients = IntegratedGradients(model)

    # Proper forward function that passes only required args to the model
    forward_func = lambda i_ids, att_mask: model(i_ids, attention_mask=att_mask)

    # Compute attributions
    attributions = integrated_gradients.attribute(
        inputs=(input_ids, attention_mask),
        baselines=(baseline_ids, baseline_mask),
        target=label,
        method="riemann_trapezoid",
        internal_batch_size=1
    )
    return attributions.squeeze(0)  # Remove batch dimension for output

# Assume input_ids and attention_mask are already defined and correctly shaped
attributions = compute_attributions(model, input_ids, attention_mask, label=0)

ValueError: too many values to unpack (expected 2)

In [13]:
import torch
import torch.nn as nn
from transformers import BertModel, XLMRobertaModel, DistilBertModel, BertTokenizer

class CustomBERTModel(nn.Module):
    def __init__(self, num_labels):
        super(CustomBERTModel, self).__init__()
        self.bert_model = BertModel.from_pretrained('bert-base-multilingual-cased')
        self.xlm_roberta_model = XLMRobertaModel.from_pretrained('xlm-roberta-base')
        self.distilbert_model = DistilBertModel.from_pretrained('distilbert-base-multilingual-cased')
        self.fc = nn.Linear(768 * 3, num_labels)

    def forward(self, input_ids, attention_mask=None, token_type_ids=None):
        bert_output = self.bert_model(input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids).last_hidden_state
        xlm_roberta_output = self.xlm_roberta_model(input_ids, attention_mask=attention_mask).last_hidden_state
        distilbert_output = self.distilbert_model(input_ids, attention_mask=attention_mask).last_hidden_state

        concatenated = torch.cat((bert_output, xlm_roberta_output, distilbert_output), dim=-1)
        out = self.fc(concatenated[:, 0, :])
        return out

# Load model
model = CustomBERTModel(num_labels=2)
model.load_state_dict(torch.load("best_model.pth"))
model.eval()

# Explaianable part
from captum.attr import IntegratedGradients

# CustomBERTModel definition remains the same

# Revised compute_attributions function with debugging
def compute_attributions(model, input_ids, attention_mask, label):
    print(f"Input IDs type before unsqueeze: {input_ids.dtype}")  # Debugging print
    input_ids = input_ids.unsqueeze(0).type(torch.LongTensor)
    print(f"Input IDs type after unsqueeze: {input_ids.dtype}")  # Debugging print
    
    if attention_mask is not None:
        attention_mask = attention_mask.unsqueeze(0).type(torch.LongTensor)

    baseline_ids = torch.zeros_like(input_ids).long()
    baseline_mask = torch.zeros_like(attention_mask).long() if attention_mask is not None else None

    label = torch.tensor([label]).long()

    integrated_gradients = IntegratedGradients(model)

    forward_func = lambda i_ids, att_mask: model(i_ids, attention_mask=att_mask, token_type_ids=None)[0]
    
    attributions = integrated_gradients.attribute(
        inputs=(input_ids, attention_mask),
        baselines=(baseline_ids, baseline_mask),
        target=label,
        additional_forward_args=None,
        method="riemann_trapezoid",
        internal_batch_size=1,
        return_convergence_delta=False
    )
    return attributions[0].squeeze(0)

# Tokenizer, model loading, and sample text processing remains the same

# Ensure the types are correct before calling compute_attributions
print(f"Input IDs type before compute_attributions: {input_ids.dtype}")  # Debugging print
input_ids = input_ids.type(torch.LongTensor)
attention_mask = attention_mask.type(torch.LongTensor)

# Compute attributions
label = 0
attributions = compute_attributions(model, input_ids[0], attention_mask[0], label)

# Visualize attributions
tokens = tokenizer.convert_ids_to_tokens(input_ids[0])
for token, attribution in zip(tokens, attributions):
    print(f"Token: {token} \t Attribution: {attribution.item()}")

Input IDs type before compute_attributions: torch.int64
Input IDs type before unsqueeze: torch.int64
Input IDs type after unsqueeze: torch.int64


RuntimeError: Expected tensor for argument #1 'indices' to have one of the following scalar types: Long, Int; but got torch.FloatTensor instead (while checking arguments for embedding)

In [ ]:
print("Input IDs type:", input_ids.dtype)  # Should output torch.int64 or torch.long


In [ ]:
print("Attention mask type:", attention_mask.dtype)  # Should output torch.int64 or torch.long


In [ ]:
input_ids = torch.tensor(input_ids).to(device).long()
